In [71]:
import torch
import torch.utils.data as data 
import torch.nn as nn
# import torch.nn.functional as F

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

## Data Loading + Processing

In [72]:
df = pd.read_csv('./data.csv', delimiter=",", index_col=0) 
df.head()

,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
instant,,,,,,,,,,,,,,,,
1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [73]:
from copy import deepcopy

input_data = deepcopy(df)
input_data = input_data.drop(columns=["casual", "registered", "cnt"])
labels = deepcopy(df["cnt"])

# print(input_data.head(), input_data.shape)
# print(labels.head())

input_data["dteday"] = pd.to_datetime(input_data["dteday"])
input_data["day"] = input_data["dteday"].dt.day
input_data.drop(columns="dteday", inplace=True)

if input_data.isna().any().any():
    for column in input_data.columns:
        input_data[column] = input_data[column].fillna(input_data[column].median())

input_data.head()

,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,day
instant,,,,,,,,,,,,,
1,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,1
2,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,1
3,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,1
4,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,1
5,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,1


In [74]:
labels.values

array([ 16,  40,  32, ..., 168, 129,  88], shape=(10886,))

In [75]:
scaler = StandardScaler()
input_data = scaler.fit_transform(input_data)
input_data

array([[-1.34986372, -1.00386564, -1.60312148, ...,  0.99321305,
        -1.56775367, -1.64206967],
       [-1.34986372, -1.00386564, -1.60312148, ...,  0.94124921,
        -1.56775367, -1.64206967],
       [-1.34986372, -1.00386564, -1.60312148, ...,  0.94124921,
        -1.56775367, -1.64206967],
       ...,
       [ 1.33801198,  0.99614925,  1.59063939, ..., -0.04606385,
         0.26970368,  1.64478707],
       [ 1.33801198,  0.99614925,  1.59063939, ..., -0.04606385,
        -0.83244247,  1.64478707],
       [ 1.33801198,  0.99614925,  1.59063939, ...,  0.21375537,
        -0.46560752,  1.64478707]], shape=(10886, 13))

In [76]:
# sklearn scaler returns ndarray from transform, dataFrame.values returns ndarray of values
train_dataset = data.TensorDataset(torch.from_numpy(input_data).float(), torch.from_numpy(labels.values).float())
next(iter(train_dataset))

(tensor([-1.3499, -1.0039, -1.6031, -1.6689, -0.1715,  1.4949, -1.4607, -0.6602,
         -1.3337, -1.0927,  0.9932, -1.5678, -1.6421]),
 tensor(16.))

In [77]:
data_loader = data.DataLoader(train_dataset, batch_size=64, shuffle=False, drop_last=False)

In [78]:
device = torch.device("mps" if torch.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

## Model definition

In [79]:
from dataclasses import dataclass


@dataclass
class HiddenLayers:
    first_layer: int = 32
    second_layer: int = 16


class Regressor(nn.Module):
    def __init__(self, input_size: int, hidden_layers: HiddenLayers = HiddenLayers()):
        super().__init__()
        self.input_size = input_size
        self.hidden_layers = hidden_layers
        self.layers = nn.Sequential(
            nn.Linear(self.input_size, self.hidden_layers.first_layer),
            nn.BatchNorm1d(self.hidden_layers.first_layer),
            nn.GELU(),
            nn.Linear(self.hidden_layers.first_layer, self.hidden_layers.second_layer),
            nn.BatchNorm1d(self.hidden_layers.second_layer),
            nn.GELU(),
            nn.Linear(self.hidden_layers.second_layer, 1),
        )

    def forward(self, input):
        return self.layers(input)


features_num = input_data.shape[1]
# features_num
model = Regressor(features_num)
model.to(device)


Regressor(
  (layers): Sequential(
    (0): Linear(in_features=13, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): GELU(approximate='none')
    (6): Linear(in_features=16, out_features=1, bias=True)
  )
)

### Train

In [80]:
EPOCHS = 100


optim = torch.optim.Adam(model.parameters(), lr=0.001)
loss_criterion = nn.MSELoss()

model.train()

for epoch in range(EPOCHS):
    for data_batch, label in data_loader:
        
        input = data_batch.to(device)
        label = label.to(device)

        y_preds = model(input)
        loss = loss_criterion(y_preds, label)
        
        optim.zero_grad()
        loss.backward()
        optim.step()

        

/Users/kwasus/Desktop/STUDIA/SEM6/SSNE/Neural-Network-Concepts/.venv/lib/python3.13/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/Users/kwasus/Desktop/STUDIA/SEM6/SSNE/Neural-Network-Concepts/.venv/lib/python3.13/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([6])) that is different to the input size (torch.Size([6, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


### Eval Data Loading + Processing

In [81]:
eval_df = pd.read_csv('./evaluation_data.csv', delimiter=",", index_col=0) 
eval_df.head()

,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed
dteday,,,,,,,,,,,,
2011-01-20,1,0,1,0,0,4,1,1,0.26,0.2273,0.56,0.3881
2011-01-20,1,0,1,1,0,4,1,1,0.26,0.2727,0.56,0.0000
2011-01-20,1,0,1,2,0,4,1,1,0.26,0.2727,0.56,0.0000
2011-01-20,1,0,1,3,0,4,1,1,0.26,0.2576,0.56,0.1642
2011-01-20,1,0,1,4,0,4,1,1,0.26,0.2576,0.56,0.1642


In [82]:
from copy import deepcopy

eval_data = deepcopy(eval_df)
eval_data = eval_data.drop(columns=["casual", "registered", "cnt"])
eval_data_labels = deepcopy(eval_df["cnt"])

# print(input_data.head(), input_data.shape)
# print(labels.head())

eval_data["dteday"] = pd.to_datetime(eval_data["dteday"])
eval_data["day"] = eval_data["dteday"].dt.day
eval_data.drop(columns="dteday", inplace=True)

if eval_data.isna().any().any():
    for column in eval_data.columns:
        eval_data[column] = eval_data[column].fillna(eval_data[column].median())

eval_data.head()

KeyError: "['casual', 'registered', 'cnt'] not found in axis"

In [70]:
eval_data = scaler.transform(eval_data)
eval_data_labels.values.shape

# eval_dataset = data.TensorDataset(torch.from_numpy(eval_data).float(), torch.from_numpy(eval_data_labels.values).float())
# eval_data_loader = data.DataLoader(eval_dataset, batch_size=64, shuffle=False, drop_last=False)

# next(iter(eval_data_loader))

NameError: name 'eval_data' is not defined

### Eval

In [ ]:
def rmsle(y_true, y_pred):
    n = len(y_true)
    msle = np.mean([(np.log(max(y_pred[i], 0) + 1) - np.log(y_true[i] + 1)) ** 2.0 for i in range(n)])
    return np.sqrt(msle)

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for eval_data, eval_labels in eval_data_loader:

        input = eval_data.to(device)
        labels = eval_labels.to(device)

        y_preds = model(input)

        all_preds.append(y_preds)
        all_labels.append(labels)
        
all_preds = torch.cat(all_preds).squeeze(1).cpu().numpy()
all_labels = torch.cat(all_labels).squeeze(1).cpu().numpy()

rmsle(all_labels, all_preds)
